In [22]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from manual_model import RandomForest, SVM

In [23]:
data = pd.read_csv("data_bersih2.csv")

In [24]:
X = data.drop(columns=['skor_risiko_korupsi', 'indikator_penawar_tunggal', "indikator_prosedur_berisiko", "indikator_submisi_berisiko", "indikator_keputusan_berisiko", "konsentrasi_instansi", 'label'])

y = data['label']

In [25]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

print("Jumlah data latih:", X_train.shape[0])
print("Jumlah data uji   :", X_test.shape[0])

Jumlah data latih: 79958
Jumlah data uji   : 19990


In [26]:
# 1. Cek kolom non-numerik
non_numeric_cols = X_train.select_dtypes(include=['object']).columns
print("Kolom non-numerik:", list(non_numeric_cols))

# 2. Drop kolom non-numerik dari data training sebelum SMOTE
X_train_num = X_train.drop(columns=non_numeric_cols)

# 3. Terapkan SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_num, y_train)

# 4. Cek hasilnya
from collections import Counter
print("Distribusi label setelah SMOTE:", Counter(y_train_resampled))

Kolom non-numerik: ['id_tender', 'judul_tender', 'jenis_prosedur', 'jenis_pengadaan', 'tanggal_kontrak', 'tanggal_keputusan_pemenang', 'status_lot', 'nama_instansi', 'negara_instansi', 'id_penyedia', 'nama_penyedia', 'tipe_penyedia', 'penyedia_menang', 'sumber_data', 'id_lot', 'id_penawaran']
Distribusi label setelah SMOTE: Counter({0: 70717, 1: 70717})


In [27]:
# 1. Gabungkan X dan y untuk mempermudah undersampling
data = pd.concat([X, pd.Series(y, name='label')], axis=1)

# 2. Pisahkan berdasarkan kelas
class_0 = data[data['label'] == 0]
class_1 = data[data['label'] == 1]

# 3. Undersample kelas mayoritas 
class_0_undersampled = class_0.sample(n=len(class_1), random_state=42)

# 4. Gabungkan kembali
data_balanced = pd.concat([class_0_undersampled, class_1], axis=0).sample(frac=1, random_state=42)  # acak data

# 5. Pisahkan kembali menjadi X dan y
X = data_balanced.drop(['id_tender', 'id_lot', 'id_penawaran', 'id_penyedia','nama_penyedia', 'judul_tender', 'nama_instansi','tanggal_kontrak', 'tanggal_keputusan_pemenang', "label"], axis=1)
y = data_balanced['label']


col_kategorikal = X.select_dtypes(include=['object']).columns

# Jika ada kolom kategorikal, lakukan encoding
if len(col_kategorikal) > 0:
    for col in col_kategorikal:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        # Transform juga X_test agar konsisten (dengan caveat: label harus sama)
        if col in X_test.columns:
            X_test[col] = le.transform(X_test[col])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [30]:
rf = RandomForest(n_trees=5, max_depth=5, min_samples_split=2, n_feature=5)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("📊 Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


📊 Random Forest Accuracy: 0.9411382817571954
              precision    recall  f1-score   support

           0       0.99      0.89      0.94      2311
           1       0.90      0.99      0.94      2310

    accuracy                           0.94      4621
   macro avg       0.95      0.94      0.94      4621
weighted avg       0.95      0.94      0.94      4621



In [31]:
# Save RF model
with open("rf_manual_model.pkl", "wb") as f:
    pickle.dump(rf, f)

In [36]:
y_train_svm = y_train.replace(0, -1)
y_test_svm = y_test.replace(0, -1)

svm = SVM(max_iterations=500, C=5)
svm.fit(X_train, y_train_svm)

y_pred_svm = svm.predict(X_test)
y_pred_svm = np.where(y_pred_svm == -1, 0, 1)

print("📊 SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

📊 SVM Accuracy: 0.6801558104306428
              precision    recall  f1-score   support

           0       0.70      0.62      0.66      2311
           1       0.66      0.74      0.70      2310

    accuracy                           0.68      4621
   macro avg       0.68      0.68      0.68      4621
weighted avg       0.68      0.68      0.68      4621



In [37]:
# Save SVM model
with open("svm_manual_model.pkl", "wb") as f:
    pickle.dump(svm, f)